# B2B Radar — restricted manual evaluation

This notebook reviews the checksum-bound stage-12 sample and writes pipeline-compatible annotations outside `ml-runs`. Source text is hidden until explicit opt-in. The review directory is restricted data, not a public aggregate report.

In [ ]:
REPOSITORY_URL = "https://github.com/osmirnov34/b2b-radar.git"
CODE_REF = "main"  # Prefer the commit used by the source run.
PROJECT_DIR = "/content/b2b-radar-manual-review"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/b2b-radar"
RUN_ID = "colab-full-001"
REVIEWER = "reviewer-name"
SHOW_PRIVATE_TEXT = False
DATA_CLASSIFICATION = "restricted_manual_review"
SAVE_ANNOTATIONS = False
OVERWRITE_ANNOTATIONS = False
OTHER_REVIEWER_ANNOTATION_FILES = []

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

project_root = Path(PROJECT_DIR)
if project_root.exists():
    raise FileExistsError(f"Manual-review checkout already exists; restart the runtime: {project_root}")
subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(project_root)], check=True)
subprocess.run(["git", "-C", str(project_root), "checkout", CODE_REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project_root)], check=True)
sys.path.insert(0, str(project_root))

In [ ]:
import pandas as pd
from google.colab import drive

from src.ml.evaluation import ManualAnnotation, ManualReviewRecord
from src.ml.experiment_passport import build_experiment_passport, write_experiment_passport
from src.ml.manual_review import (
    load_manual_review_bundle,
    reviewer_agreement,
    save_manual_annotations,
    summarize_manual_annotations,
    validate_manual_annotations,
)
from src.ml.reporting import explain_assignment, load_analysis_artifacts

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,79}", REVIEWER):
    raise ValueError("REVIEWER must be a safe non-empty identifier")
drive.mount("/content/drive")
run_dir = Path(DRIVE_PROJECT_DIR) / "ml-runs" / RUN_ID
review_dir = Path(DRIVE_PROJECT_DIR) / "manual-reviews" / RUN_ID / REVIEWER
annotations_path = review_dir / "manual-annotations.jsonl"
bundle = load_manual_review_bundle(run_dir, annotations_path if annotations_path.is_file() else None)
artifacts = load_analysis_artifacts(run_dir)
passport = build_experiment_passport(run_dir, project_root, code_ref=CODE_REF)
annotations_by_index = {item.record_index: item for item in bundle.annotations}
display(pd.DataFrame([summarize_manual_annotations(bundle).model_dump(mode="json")]))
display(
    pd.DataFrame(
        [
            {
                "run_id": passport.run_id,
                "pipeline_status": passport.pipeline_status,
                "git_commit": passport.git_commit,
                "git_dirty": passport.git_dirty,
                "manifests": len(passport.manifests),
                "completed_stages": passport.completed_stages,
            }
        ]
    )
)

In [ ]:
def show_review_item(record_index: int) -> ManualReviewRecord:
    """Display one verified sample row and its assignment context after privacy opt-in."""
    if not SHOW_PRIVATE_TEXT:
        raise PermissionError("Set SHOW_PRIVATE_TEXT=True before displaying restricted review text")
    sample = next((item for item in bundle.sample if item.record_index == record_index), None)
    if sample is None:
        raise KeyError(f"Record {record_index} is not in the verified review sample")
    explanation = explain_assignment(artifacts, record_index, representative_limit=3)
    display(
        pd.DataFrame(
            [
                {
                    "record_index": sample.record_index,
                    "topic_id": sample.topic_id,
                    "sample_kind": sample.sample_kind,
                    "confidence": sample.confidence,
                    "text": sample.text,
                    "video_url": explanation.video_url,
                    "topic_name": explanation.topic_name,
                    "topic_keywords": " | ".join(explanation.topic_keywords),
                }
            ]
        ).style.format(hyperlinks="html")
    )
    return sample


def set_annotation(
    record_index: int,
    *,
    topic_matches: bool,
    topic_clear: bool,
    business_relevant: bool,
    reassignment_correct: bool | None = None,
    contains_sensitive_data: bool = False,
    merge_candidate: bool = False,
    split_candidate: bool = False,
    note: str = "",
) -> ManualAnnotation:
    """Validate and stage one pipeline-compatible judgment in notebook memory."""
    annotation = ManualAnnotation(
        record_index=record_index,
        topic_matches=topic_matches,
        topic_clear=topic_clear,
        business_relevant=business_relevant,
        reassignment_correct=reassignment_correct,
        contains_sensitive_data=contains_sensitive_data,
        merge_candidate=merge_candidate,
        split_candidate=split_candidate,
        reviewer=REVIEWER,
        note=note,
    )
    validate_manual_annotations(bundle.sample, [annotation])
    annotations_by_index[record_index] = annotation
    return annotation


def next_unreviewed_record_index() -> int | None:
    """Return the next deterministic sample index absent from current annotations."""
    return next((item.record_index for item in bundle.sample if item.record_index not in annotations_by_index), None)


print({"next_unreviewed_record_index": next_unreviewed_record_index()})

## Review loop

Set `SHOW_PRIVATE_TEXT=True`, call `show_review_item(next_unreviewed_record_index())`, then record the judgment with `set_annotation(...)`. For a `reassigned` sample, `reassignment_correct` must be `True` or `False`; for other sample kinds it must remain `None`. Use `contains_sensitive_data=True` whenever the displayed text exposes personal data.

In [ ]:
# Example — edit values and remove the leading # only after reviewing the displayed record.
# record_index = next_unreviewed_record_index()
# show_review_item(record_index)
# set_annotation(
#     record_index,
#     topic_matches=True,
#     topic_clear=True,
#     business_relevant=True,
#     reassignment_correct=None,
#     note="",
# )

In [ ]:
current_annotations = validate_manual_annotations(bundle.sample, list(annotations_by_index.values()))
current_bundle = bundle.model_copy(update={"annotations": current_annotations})
display(pd.DataFrame([summarize_manual_annotations(current_bundle).model_dump(mode="json")]))
reviewer_sets = {REVIEWER: current_annotations}
for reviewer_file in OTHER_REVIEWER_ANNOTATION_FILES:
    other_bundle = load_manual_review_bundle(run_dir, Path(reviewer_file))
    if other_bundle.annotations:
        reviewer_sets[other_bundle.annotations[0].reviewer] = other_bundle.annotations
agreement = reviewer_agreement(reviewer_sets)
display(pd.DataFrame([agreement.model_dump(mode="json")]))

In [ ]:
if SAVE_ANNOTATIONS:
    annotation_manifest = save_manual_annotations(
        current_bundle, current_annotations, review_dir, overwrite=OVERWRITE_ANNOTATIONS
    )
    passport_path = write_experiment_passport(passport, review_dir, run_dir, overwrite=OVERWRITE_ANNOTATIONS)
    print(
        {
            "annotations": annotation_manifest.annotations_path,
            "annotations_sha256": annotation_manifest.annotations_sha256,
            "experiment_passport": str(passport_path),
            "classification": annotation_manifest.classification,
        }
    )
else:
    print("Nothing saved. Set SAVE_ANNOTATIONS=True only after reviewing the in-memory summary.")